In [1]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from keras.models import Model
from keras.layers import Input, Dense
from keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix

In [2]:
caminho_pasta_tratado = '../../dataset tratado/cicids2017/Sem Redução de Dimensionalidade/'

nome_dados_treinamento = 'cicids2017_treinamento.csv'
nome_dados_teste       = 'cicids2017_teste.csv'

In [3]:
# Carregando o dataset de treinamento e filtrando apenas tráfego BENIGNO
print("Carregando dataset de treinamento...")
df_treino_full = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
print(f"Dataset completo: {df_treino_full.shape}")

df_treino_benign = df_treino_full[df_treino_full['Label'] == 'BENIGN'].copy()
print(f"Apenas BENIGN: {df_treino_benign.shape}")

X_treino = df_treino_benign.drop('Label', axis=1).values

display(df_treino_benign.head())

Carregando dataset de treinamento...
Dataset completo: (1979513, 81)
Apenas BENIGN: (1590306, 81)


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.006760,0.855604,0.352941,3.229166e-05,0.000009,0.000003,2.945736e-06,9.153974e-09,0.001531,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,0.625956,0.006760,0.352941,2.272266e-03,0.000009,0.000017,0.000000e+00,4.576987e-09,0.000000,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,0.006760,0.848402,0.352941,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,0.801816,0.000809,1.000000,1.233333e-06,0.000005,0.000007,6.821705e-06,1.830795e-07,0.001773,0.018925,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,0.583032,0.000809,1.000000,1.449567e-03,0.000005,0.000007,4.496124e-06,3.509023e-07,0.001168,0.012473,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [4]:
# Arquitetura do Autoencoder
# Encoder: 78 → 64 → 32 → 16
# Decoder: 16 → 32 → 64 → 78
n_features = X_treino.shape[1]

inputs = Input(shape=(n_features,))

# Encoder
encoded = Dense(64, activation='relu')(inputs)
encoded = Dense(32, activation='relu')(encoded)
encoded = Dense(16, activation='relu')(encoded)

# Decoder
decoded = Dense(32, activation='relu')(encoded)
decoded = Dense(64, activation='relu')(decoded)
outputs = Dense(n_features, activation='sigmoid')(decoded)

autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         5,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 80)             │         5,200 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,648 (61.12 KB)

 Trainable params: 15,648 (61.12 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Treinamento do Autoencoder (apenas em tráfego BENIGN)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Iniciando treinamento do Autoencoder...")
inicio_treino = time.time()

history = autoencoder.fit(
    X_treino, X_treino,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

fim_treino = time.time()
print(f"\nTreinamento concluído! Tempo total: {fim_treino - inicio_treino:.2f} segundos.")

Iniciando treinamento do Autoencoder...
Epoch 1/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.0022 - val_loss: 2.3814e-04
Epoch 2/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 6s 987us/step - loss: 1.9005e-04 - val_loss: 1.1892e-04
Epoch 3/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 6s 982us/step - loss: 9.4385e-05 - val_loss: 7.9001e-05
Epoch 4/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 6s 987us/step - loss: 7.1997e-05 - val_loss: 6.3944e-05
Epoch 5/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 6s 978us/step - loss: 6.0512e-05 - val_loss: 5.6058e-05
Epoch 6/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 6s 982us/step - loss: 5.3477e-05 - val_loss: 5.0541e-05
Epoch 7/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 6s 984us/step - loss: 4.8217e-05 - val_loss: 4.7169e-05
Epoch 8/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 6s 988us/step - loss: 4.1462e-05 - val_loss: 3.6277e-05
Epoch 9/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 6s 991us/step - loss: 3.7061e-05 - val_loss: 3.6504e-05
Epoch 10/50
5591/5591 ━━━━━━━━━━━━━━━━━━━━ 5s 974us/step - loss: 3.4935e-05 - val_lo

In [6]:
# Definindo o limiar de anomalia
# Usamos o percentil 95 do erro de reconstrução no conjunto de treino BENIGN
X_treino_rec = autoencoder.predict(X_treino, verbose=0)
erros_treino = np.mean(np.power(X_treino - X_treino_rec, 2), axis=1)

PERCENTIL = 95
limiar = np.percentile(erros_treino, PERCENTIL)

print(f"Estatísticas do erro de reconstrução (BENIGN treino):")
print(f"  Média:   {erros_treino.mean():.6f}")
print(f"  Desvio:  {erros_treino.std():.6f}")
print(f"  Máximo:  {erros_treino.max():.6f}")
print(f"  Limiar ({PERCENTIL}º percentil): {limiar:.6f}")

Estatísticas do erro de reconstrução (BENIGN treino):
  Média:   0.000018
  Desvio:  0.000606
  Máximo:  0.088828
  Limiar (95º percentil): 0.000036


In [7]:
def avaliar_autoencoder(autoencoder, limiar, df_teste, nome_cenario, benign_label):
    X_teste = df_teste.drop('Label', axis=1).values
    y_teste_real = df_teste['Label'].values

    X_teste_rec = autoencoder.predict(X_teste, verbose=0)
    erros_teste = np.mean(np.power(X_teste - X_teste_rec, 2), axis=1)

    y_pred_binario = np.where(erros_teste > limiar, 'ATTACK', 'BENIGN')
    y_real_binario = np.where(y_teste_real == benign_label, 'BENIGN', 'ATTACK')

    labels_bin = ['BENIGN', 'ATTACK']
    cm = confusion_matrix(y_real_binario, y_pred_binario, labels=labels_bin)
    cm_df = pd.DataFrame(
        cm,
        index=[f"Real_{l}" for l in labels_bin],
        columns=[f"Pred_{l}" for l in labels_bin]
    )

    print(f"\n{'='*60}")
    print(f"CENÁRIO: {nome_cenario}")
    print(f"Total de amostras: {len(y_teste_real)}")
    print(f"  BENIGN: {(y_real_binario == 'BENIGN').sum()}")
    print(f"  ATTACK: {(y_real_binario == 'ATTACK').sum()}")
    print(f"\n=== MATRIZ DE CONFUSÃO ===")
    display(cm_df.style.format("{:.0f}"))
    print(f"\n=== RELATÓRIO DE MÉTRICAS ===")
    print(classification_report(y_real_binario, y_pred_binario, labels=labels_bin, zero_division=0))

    return y_real_binario, y_pred_binario, cm

In [8]:
CLASS_ALIASES_LATEX = {'BENIGN': 'BENIGN', 'ATTACK': 'ATTACK'}


def escape_latex(value):
    replacements = {
        "\\": "\\textbackslash{}",
        "&": "\\&",
        "%": "\\%",
        "$": "\\$",
        "#": "\\#",
        "_": "\\_",
        "{": "\\{",
        "}": "\\}",
        "~": "\\textasciitilde{}",
        "^": "\\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return f"\\ok{{{value}}}"
    if value != 0:
        return f"\\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [escape_latex(CLASS_ALIASES_LATEX.get(l, l)) for l in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((f"Real\\_{escape_latex(CLASS_ALIASES_LATEX.get(real_label, real_label))}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        f"        \\begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        "            \\hline",
        format_row("", headers),
        "            \\hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        "            \\hline",
        "        \\end{tabular}",
        "    }",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values, y_pred_values,
        labels=class_labels, output_dict=True, zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            escape_latex(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        ["\\textbf{Acurácia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        ["\\textbf{Média Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        ["\\textbf{Média Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precisão", "Revocação", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + " \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        "    \\begin{tabular}{lrrrr}",
        "        \\hline",
        format_row(headers),
        "        \\hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        "        \\hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        "        \\hline",
        "    \\end{tabular}",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)

In [9]:
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)
y_real, y_pred, cm = avaliar_autoencoder(autoencoder, limiar, df_teste, 'Teste Completo', 'BENIGN')

labels_bin = ['BENIGN', 'ATTACK']

print(make_latex_confusion_matrix(
    cm, labels_bin,
    'Autoencoder — CICIDS2017 (Teste Completo) — Matriz de Confusão',
    'table:ae_cicids_completo_mc',
))
print()
print(make_latex_metrics_report(
    y_real, y_pred, labels_bin,
    'Autoencoder — CICIDS2017 (Teste Completo) — Relatório de Métricas',
    'table:ae_cicids_completo_metricas',
))


CENÁRIO: Teste Completo
Total de amostras: 848363
  BENIGN: 681014
  ATTACK: 167349

=== MATRIZ DE CONFUSÃO ===


,Pred_BENIGN,Pred_ATTACK
Real_BENIGN,646544,34470
Real_ATTACK,86742,80607



=== RELATÓRIO DE MÉTRICAS ===
              precision    recall  f1-score   support

      BENIGN       0.88      0.95      0.91    681014
      ATTACK       0.70      0.48      0.57    167349

    accuracy                           0.86    848363
   macro avg       0.79      0.72      0.74    848363
weighted avg       0.85      0.86      0.85    848363

\begin{table}[H]
    \centering
    \small
        \begin{tabular}{l|rr}
            \hline
                         & BENIGN      & ATTACK      \\
            \hline
            Real\_BENIGN & \ok{646544} & \err{34470} \\
            Real\_ATTACK & \err{86742} & \ok{80607}  \\
            \hline
        \end{tabular}
    }
    \caption{Autoencoder — CICIDS2017 (Teste Completo) — Matriz de Confusão}
    \label{table:ae_cicids_completo_mc}
\end{table}

\begin{table}[H]
    \centering
    \small
    \begin{tabular}{lrrrr}
        \hline
        Classe                   & Precisão & Revocação & F1-score & Suporte \\
        \hline
      